In [ ]:
import sys
from pathlib import Path

for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
    if (candidate / 'dataset').exists() and (candidate / 'paths.py').exists():
        candidate_str = str(candidate.resolve())
        if candidate_str not in sys.path:
            sys.path.insert(0, candidate_str)
        break


### Imports

In [ ]:
from config import MEDICAL_ASSISTANT_MODEL, MEDICAL_ASSISTANT_TEMPERATURE, PATIENT_ASSISTANT_MODEL, PATIENT_ASSISTANT_TEMPERATURE
from config import EVALUATION_MODE, DRY_RUN, MAX_SAMPLES, MAX_EVAL_PARALLEL_SIZE
from config import SELECTED_HADM_IDS
from config import DATASET_NAMES

from runs.run import run_simulations
from runs.run import tools, tools_for_planning_routine, func_name_to_func
from dataset.mimic_dataset import MIMIC_Dataset
from backend.fhir_client import base_url, headersList, session
from paths import EVALUABLE_OUTPUTS_BASELINE_DIR

from routines import COMPLETION_PROMPT, PATIENT_SYSTEM_PROMPT

import warnings
warnings.filterwarnings("ignore")

OUTPUT_DIR = str(EVALUABLE_OUTPUTS_BASELINE_DIR)
# DATASET_NAMES = ["appendicitis"]  # choose one from config.SELECTED_HADM_IDS

MAX_EVAL_PARALLEL_SIZE = 1  # no parallelism
MAX_SAMPLES = 1


### Start the Run

In [ ]:
for DATASET_NAME in DATASET_NAMES:

    if DATASET_NAME == "pancreatic_cancer":
        from routines import (
            MEDICAL_SYSTEM_PROMPT_PANCREATIC_CANCER as MEDICAL_SYSTEM_PROMPT,
        )
    else:
        from routines import MEDICAL_SYSTEM_PROMPT

    selected_hadm_ids = SELECTED_HADM_IDS[DATASET_NAME]

    ds = MIMIC_Dataset.load_dataset(DATASET_NAME)
    _ = run_simulations(
        output_dir=OUTPUT_DIR,
        ds_name=DATASET_NAME,
        ds=ds,
        base_url=base_url,
        headers_list=headersList,
        session=session,
        tools_for_planning_routine=tools_for_planning_routine,
        tools=tools,
        func_name_to_func=func_name_to_func,
        medical_assistant_model=MEDICAL_ASSISTANT_MODEL,
        medical_assistant_temperature=MEDICAL_ASSISTANT_TEMPERATURE,
        patient_assistant_model=PATIENT_ASSISTANT_MODEL,
        patient_assistant_temperature=PATIENT_ASSISTANT_TEMPERATURE,
        medical_system_prompt=MEDICAL_SYSTEM_PROMPT,
        completion_prompt=COMPLETION_PROMPT,
        patient_system_prompt=PATIENT_SYSTEM_PROMPT,
        max_samples=MAX_SAMPLES,
        evaluation_mode=EVALUATION_MODE,
        max_workers=MAX_EVAL_PARALLEL_SIZE,
        dry_run=DRY_RUN,
        selected_hadm_ids=selected_hadm_ids,
        save_dir=OUTPUT_DIR,
    )